# Балансировка коленчатого вала — демонстрация

Ноутбук использует пакет `crankshaft-balancing` (`pip install -e .` из корня репозитория).
Вся логика находится в `src/crankshaft_balancing/`, здесь — только вызовы и графики.
Теория — в `docs/theory.md`.

In [ ]:
import matplotlib.pyplot as plt
from crankshaft_balancing import (analyze, load_shaft, make_initial_coefficients,
                                  sweep_sensitivity, verify_kkt_conditions,
                                  verify_optimum_via_duality, NMM_TO_NM)
from crankshaft_balancing.analysis import raw_moments
from crankshaft_balancing.plotting import (plot_distribution, plot_moment_3d,
                                           plot_sensitivity)

## 1. Исходные данные и расчёт
Замените `example_shaft.json` на свой файл — формат описан в README.

In [ ]:
engine, components = load_shaft("../examples/example_shaft.json")
res = analyze(engine, components, step=0.01)
f, Z, L = res.forces, res.Z, res.engine.crankshaft_length

## 2. Эпюры момента до оптимизации

In [ ]:
Mx_raw, My_raw = raw_moments(Z, f)
groups = lambda Fc, Fp: [{"L": f.L_cheeks, "F": Fc, "name": "Cheek"},
                         {"L": f.L_crankpins, "F": Fp, "name": "crankpin"}]
plot_distribution(Z, Mx_raw, L, groups=groups(f.Fx_cheeks, f.Fx_crankpins),
                  title="Момент M(z) до оптимизации (ось X)", ylabel="Момент, Н·м")
plot_distribution(Z, My_raw, L, groups=groups(f.Fy_cheeks, f.Fy_crankpins),
                  title="Момент M(z) до оптимизации (ось Y)", ylabel="Момент, Н·м")

## 3. После подбора противовесов

In [ ]:
for axis, r, Fc, Fp in (("Y", res.result_y, f.Fy_cheeks, f.Fy_crankpins),
                        ("X", res.result_x, f.Fx_cheeks, f.Fx_crankpins)):
    plot_distribution(
        Z, r["M"] * NMM_TO_NM, L,
        groups=groups(Fc, Fp) + [{"L": f.L_cheeks, "F": r["F3"], "name": "F3 (оптим.)"}],
        title=f"Момент M(z) после оптимизации F3 (ось {axis})", ylabel="Момент, Н·м")

## 4. Суммарный момент

In [ ]:
plot_moment_3d(Z, res.result_x, res.result_y, shaft_length=L)
# Интерактивная версия (нужен plotly):
# plot_moment_3d(Z, res.result_x, res.result_y, shaft_length=L,
#                interactive=True, save_path="moment_3d.html")

## 5. Проверка, что минимум глобальный

In [ ]:
for axis, r in (("y", res.result_y), ("x", res.result_x)):
    verify_optimum_via_duality(r, axis=axis)
    verify_kkt_conditions(r, axis=axis)
    print()

In [ ]:
curves = sweep_sensitivity(res.result_y, Z, f.L_cheeks, f.Fy_cheeks,
                           f.L_crankpins, f.Fy_crankpins, engine.angular_speed)
plot_sensitivity(curves, res.result_y["s"], axis="y")

## 6. Сходимость симплекс-метода (необязательно)

Использует устаревший `linprog(method="simplex")` — доступен не во всех версиях SciPy
и работает на грубой сетке (шаг 2 мм).

In [ ]:
from crankshaft_balancing import optimize_counterweights_history, section_grid
from crankshaft_balancing.plotting import plot_convergence

Z_coarse = section_grid(L, step=2.0)
hist = optimize_counterweights_history(
    Z_coarse, f.L_cheeks, f.Fy_cheeks, f.L_crankpins, f.Fy_crankpins,
    engine.angular_speed, make_initial_coefficients(len(f.Fy_cheeks)), axis="y")
plot_convergence(hist["history"], axis="y")